**Suppose SQL gives us:**

| date       | product_id | sales |
| ---------- | ---------- | ----: |
| 2026-01-01 | 101        |   100 |
| 2026-01-02 | 101        |   120 |
| 2026-01-03 | 101        |   110 |


In [2]:
import pandas as pd

df = pd.DataFrame({
    "date": ["2026-01-01", "2026-01-02", "2026-01-03"],
    "product_id": [101, 101, 101],
    "sales": [100, 120, 110]
})

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   date        3 non-null      str  
 1   product_id  3 non-null      int64
 2   sales       3 non-null      int64
dtypes: int64(2), str(1)
memory usage: 204.0 bytes


In [3]:
df

,date,product_id,sales
0,2026-01-01,101,100
1,2026-01-02,101,120
2,2026-01-03,101,110


In [4]:
df.dtypes

date            str
product_id    int64
sales         int64
dtype: object

In [5]:
# date: object (string) - Pandas doesn't yet know that this column represents dates.
df['date'] = pd.to_datetime(df['date'])

In [6]:
df.dtypes

date          datetime64[us]
product_id             int64
sales                  int64
dtype: object


* date: datetime64[us]
* Why does this matter?
* Because once Pandas knows something is datetime, you can do things like:

Remeber

Monday    → 0
Tuesday   → 1
...
Sunday    → 6

In [7]:
df["day_of_week"] = df["date"].dt.dayofweek
df



,date,product_id,sales,day_of_week
0,2026-01-01,101,100,3
1,2026-01-02,101,120,4
2,2026-01-03,101,110,5


In [8]:
# Sorting by time
df = df.sort_values("date")
df

,date,product_id,sales,day_of_week
0,2026-01-01,101,100,3
1,2026-01-02,101,120,4
2,2026-01-03,101,110,5


In [9]:
df["lag_1"] = df["sales"].shift(1)
df

,date,product_id,sales,day_of_week,lag_1
0,2026-01-01,101,100,3,NaN
1,2026-01-02,101,120,4,100.0
2,2026-01-03,101,110,5,120.0


This is the Pandas equivalent of:

```sql
LAG(sales) OVER (
    ORDER BY date
)
```

In [ ]:
# Part 3 - Datetime Index
df.reset_index(drop=True, inplace=True)
df = df.set_index('date')

In [11]:
df

,product_id,sales,day_of_week,lag_1
date,,,,
2026-01-01,101,100,3,NaN
2026-01-02,101,120,4,100.0
2026-01-03,101,110,5,120.0


In [12]:
weekly_sales = df["sales"].resample("W").sum()

In [13]:
weekly_sales

date
2026-01-04    330
Freq: W-SUN, Name: sales, dtype: int64

In [14]:
monthly_sales = df['sales'].resample("ME").sum()
monthly_sales

date
2026-01-31    330
Freq: ME, Name: sales, dtype: int64

In [15]:
daily_sales = df["sales"].resample("D").sum()
daily_sales

date
2026-01-01    100
2026-01-02    120
2026-01-03    110
Freq: D, Name: sales, dtype: int64

**PostgreSQL**
```sql
SELECT
    DATE_TRUNC('month', order_date) AS month,
    SUM(quantity) AS sales
FROM orders
GROUP BY DATE_TRUNC('month', order_date);
```

**Pandas**
```python
monthly_sales = (
    df["sales"]
    .resample("ME")
    .sum()
)
```

In [16]:
df = pd.DataFrame({
    "date": [
        "2026-01-01", "2026-01-01",
        "2026-01-02", "2026-01-02",
        "2026-01-03", "2026-01-03"
    ],
    "product_id": [101, 102, 101, 102, 101, 102],
    "sales": [100, 200, 120, 220, 110, 210]
})


In [ ]:
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date')
df.reset_index(drop=True, inplace=True)
df = df.set_index('date')
df

,product_id,sales
date,,
2026-01-01,101,100
2026-01-01,102,200
2026-01-02,101,120
2026-01-02,102,220
2026-01-03,101,110
2026-01-03,102,210


In [19]:
weekly_sales = (
    df
    .groupby('product_id')['sales']
    .resample('W')
    .sum()
)

weekly_sales

product_id  date      
101         2026-01-04    330
102         2026-01-04    630
Name: sales, dtype: int64

**PostgreSQL**
```sql
SELECT
    DATE_TRUNC('week', order_date) AS week,
    product_id,
    SUM(quantity) AS sales
FROM orders
GROUP BY
    DATE_TRUNC('week', order_date),
    product_id;
```

**Pandas**
```python
weekly_sales = (
    df
    .groupby("product_id")["sales"]
    .resample("W")
    .sum()
)
```

In [20]:
weekly_sales =  weekly_sales.reset_index()
weekly_sales

,product_id,date,sales
0,101,2026-01-04,330
1,102,2026-01-04,630


In [21]:
df = pd.DataFrame({
    "date": ["Jan 1", "Jan 2", "Jan 3", "Jan 4"],
    "sales": [100, 120, 110, 130]
})

df

,date,sales
0,Jan 1,100
1,Jan 2,120
2,Jan 3,110
3,Jan 4,130


In [22]:
df['lag_1'] = df['sales'].shift(1)
df

,date,sales,lag_1
0,Jan 1,100,NaN
1,Jan 2,120,100.0
2,Jan 3,110,120.0
3,Jan 4,130,110.0



Exactly like:
```sql
LAG(sales, 1) OVER (
    ORDER BY date
)
```

In [23]:
df = pd.DataFrame({
    "date": ["Jan 1", "Jan 1", "Jan 2", "Jan 2"],
    "product_id": [101, 102, 101, 102],
    "sales": [100, 200, 120, 220]
})

df

,date,product_id,sales
0,Jan 1,101,100
1,Jan 1,102,200
2,Jan 2,101,120
3,Jan 2,102,220


In [27]:
df["lag_1"] = (
    df
    .groupby("product_id")["sales"]
    .shift(1)
)

df

,date,product_id,sales,lag_1
0,Jan 1,101,100,NaN
1,Jan 1,102,200,NaN
2,Jan 2,101,120,100.0
3,Jan 2,102,220,200.0


SQL:

PARTITION BY product_id
       ↓
LAG(sales)


Pandas:

groupby("product_id")
       ↓
shift()

This connection is extremely important.

In [28]:
df

,date,product_id,sales,lag_1
0,Jan 1,101,100,NaN
1,Jan 1,102,200,NaN
2,Jan 2,101,120,100.0
3,Jan 2,102,220,200.0


In [ ]:
# Now Pandas equivalent of our SQL rolling average.
# By default it includes the current row.
df['rolling_mean_3'] = (
    df['sales']
    .rolling(3)
    .mean()
)

df

,date,product_id,sales,lag_1,rolling_mean_3
0,Jan 1,101,100,NaN,NaN
1,Jan 1,102,200,NaN,NaN
2,Jan 2,101,120,100.0,140.0
3,Jan 2,102,220,200.0,180.0


In [ ]:
# ROWS BETWEEN 3 PRECEDING AND 1 PRECEDING
df["rolling_mean_3"] = (
    df["sales"]
    .shift(1)
    .rolling(3)
    .mean()
)

df

,date,product_id,sales,lag_1,rolling_mean_3
0,Jan 1,101,100,NaN,NaN
1,Jan 1,102,200,NaN,NaN
2,Jan 2,101,120,100.0,NaN
3,Jan 2,102,220,200.0,140.0


In [31]:
dates = pd.date_range(
    start='2026-01-01',
    end='2026-01-06',
    freq='D'
)

dates

DatetimeIndex(['2026-01-01', '2026-01-02', '2026-01-03', '2026-01-04',
               '2026-01-05', '2026-01-06'],
              dtype='datetime64[us]', freq='D')

**This is roughly the Pandas equivalent of PostgreSQL:**
```sql
generate_series(
    DATE '2026-01-01',
    DATE '2026-01-06',
    INTERVAL '1 day'
)
```

In [40]:
# Let df with missing date
df = pd.DataFrame({
    "date": pd.to_datetime([
        "2026-01-01",
        "2026-01-02",
        "2026-01-03",
        "2026-01-05",
        "2026-01-06"
    ]),
    "sales": [100, 120, 110, 130, 140]
})

df

,date,sales
0,2026-01-01,100
1,2026-01-02,120
2,2026-01-03,110
3,2026-01-05,130
4,2026-01-06,140


In [41]:
df.reset_index(drop=True, inplace=True)
df.set_index('date', inplace=True)
df

,sales
date,
2026-01-01,100
2026-01-02,120
2026-01-03,110
2026-01-05,130
2026-01-06,140


In [42]:
full_dates = pd.date_range(
    start=df.index.min(),
    end=df.index.max(),
    freq="D"
)

full_dates

DatetimeIndex(['2026-01-01', '2026-01-02', '2026-01-03', '2026-01-04',
               '2026-01-05', '2026-01-06'],
              dtype='datetime64[us]', freq='D')

In [44]:
df = df.reindex(full_dates)
df

,sales
2026-01-01,100.0
2026-01-02,120.0
2026-01-03,110.0
2026-01-04,NaN
2026-01-05,130.0
2026-01-06,140.0


In [45]:
df["sales"] = df["sales"].fillna(0)
df

,sales
2026-01-01,100.0
2026-01-02,120.0
2026-01-03,110.0
2026-01-04,0.0
2026-01-05,130.0
2026-01-06,140.0


In [48]:
sales = pd.DataFrame({
    "date": ["Jan 1", "Jan 2"],
    "product_id": [101, 101],
    "sales": [100, 120]
})

products = pd.DataFrame({
    "product_id": [101],
    "category": ["Electronics"]
})

display(sales)


display(products)

,date,product_id,sales
0,Jan 1,101,100
1,Jan 2,101,120


,product_id,category
0,101,Electronics


In [49]:
df = sales.merge(
    products,
    on='product_id',
    how='left'
)

display(df)

,date,product_id,sales,category
0,Jan 1,101,100,Electronics
1,Jan 2,101,120,Electronics


```sql
SELECT ...
FROM sales AS s
LEFT JOIN products AS p
    ON s.product_id = p.product_id;
```


**SQL ↔ Pandas mapping**
SQL                         Pandas

INNER JOIN             →    merge(how="inner")

LEFT JOIN              →    merge(how="left")

RIGHT JOIN             →    merge(how="right")

FULL OUTER JOIN        →    merge(how="outer")

| PostgreSQL           | Pandas         |
| -------------------- | -------------- |
| `DATE_TRUNC()`       | `resample()`   |
| `GROUP BY`           | `groupby()`    |
| `LAG()`              | `shift()`      |
| `LEAD()`             | `shift(-1)`    |
| Rolling window       | `rolling()`    |
| `generate_series()`  | `date_range()` |
| `LEFT JOIN` calendar | `reindex()`    |
